# Solutions — Map/Shuffle/Reduce, Combiners, Immutable Transformations, Join Strategies, Salting

Exercise notebook: `12_pyspark_preparation/01_functional_partition_thinking.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Word count as explicit map → shuffle → reduce

**Working solution**

In [ ]:
from collections import defaultdict

def map_phase(lines):
    for line in lines:
        for raw in line.split():
            word = raw.strip(".,!?").lower()
            if word:
                yield word, 1

def shuffle_phase(pairs):
    groups = defaultdict(list)
    for key, value in pairs:
        groups[key].append(value)
    return dict(groups)

def reduce_phase(groups):
    return {key: sum(values) for key, values in groups.items()}

**Explanation**

The three phases of MapReduce — and of `rdd.flatMap(...).map(lambda w: (w, 1)).reduceByKey(add)`. The shuffle is the expensive part: it moves every pair across the network
grouped by key. Everything else in this notebook is about making the shuffle smaller or avoiding it.

**Complexity:** O(words).

**Common mistakes**

- Thinking of the shuffle as free — on a cluster it's network + disk.

## Exercise 2 — 🟡 Medium · Combiners shrink the shuffle

**Working solution**

In [ ]:
import zlib
from collections import defaultdict

def partitioned_sum(records, key, value, num_partitions, use_combiner=True):
    inputs = [records[i::num_partitions] for i in range(num_partitions)]
    emitted = []
    for part in inputs:
        if use_combiner:
            local = defaultdict(float)
            for r in part:
                local[r[key]] += r[value]
            emitted.extend(local.items())
        else:
            emitted.extend((r[key], r[value]) for r in part)
    reducers = [defaultdict(float) for _ in range(num_partitions)]
    for k, v in emitted:
        reducers[zlib.crc32(str(k).encode()) % num_partitions][k] += v
    totals = {k: v for red in reducers for k, v in red.items()}
    return dict(sorted(totals.items())), len(emitted)

**Explanation**

With a combiner, shuffle size is bounded by (#partitions × #distinct keys) instead of #records. That's why `reduceByKey` beats `groupByKey`, and why `groupBy().agg(sum)` in Spark does a
partial aggregation before the exchange (look for `HashAggregate → Exchange → HashAggregate` in `explain()`).

**Complexity:** O(n).

**Common mistakes**

- `groupByKey().mapValues(sum)` — ships every value across the network.

## Exercise 3 — 🟡 Medium · Immutable DataFrame-style transformations

**Working solution**

In [ ]:
def transform(rows, steps):
    current = rows
    for step in steps:
        kind, *args = step
        if kind == "with_column":
            name, fn = args
            current = [{**r, name: fn(r)} for r in current]
        elif kind == "filter":
            (pred,) = args
            current = [r for r in current if pred(r)]
        elif kind == "select":
            (cols,) = args
            current = [{c: r[c] for c in cols} for r in current]
        elif kind == "drop":
            (cols,) = args
            current = [{k: v for k, v in r.items() if k not in cols} for r in current]
        elif kind == "rename":
            (mapping,) = args
            current = [{mapping.get(k, k): v for k, v in r.items()} for r in current]
        else:
            raise ValueError(f"unknown step {kind!r}")
    return [dict(r) for r in current]

**Explanation**

Each step returns brand-new dicts, so earlier results (and the caller's data) are never changed — Spark DataFrames are immutable for the same reason: lineage can be replayed after
failures. `kind, *args = step` unpacks variable-length step tuples.

**Complexity:** O(n × steps).

**Common mistakes**

- `r[name] = fn(r)` inside `with_column` → mutates the caller's rows.

## Exercise 4 — 🟠 Hard · Broadcast join vs shuffle join

**Working solution**

In [ ]:
import zlib
from collections import defaultdict

def _local_hash_join(left, right, key):
    index = defaultdict(list)
    for r in right:
        index[r[key]].append(r)
    return [{**l, **r} for l in left for r in index.get(l[key], [])]

def broadcast_join(large, small, key):
    return _local_hash_join(large, small, key)

def shuffle_join(left, right, key, n):
    def part(rows):
        buckets = [[] for _ in range(n)]
        for r in rows:
            buckets[zlib.crc32(str(r[key]).encode()) % n].append(r)
        return buckets
    lparts, rparts = part(left), part(right)
    outputs = [_local_hash_join(lp, rp, key) for lp, rp in zip(lparts, rparts)]
    return [row for out in outputs for row in out], [len(o) for o in outputs]

**Explanation**

Co-partitioning by key guarantees matching rows meet in the same partition, so partitions can be joined independently (in parallel). Broadcasting avoids shuffling the big side when one
side is small — Spark does this automatically below `spark.sql.autoBroadcastJoinThreshold` (10 MB default) or when you call `F.broadcast(df)`.

**Complexity:** Broadcast O(n + m) with m in memory everywhere; shuffle O(n + m) network.

**Common mistakes**

- Broadcasting a 5 GB table → executor OOM.

## Challenge — 🔴 Very Hard · Fix a skewed join with salting

**Working solution**

In [ ]:
import zlib
from collections import defaultdict

def salted_join(left, right, key, n, hot_keys, salt_buckets=8):
    def row_salt(row):
        return zlib.crc32(repr(sorted(row.items())).encode()) % salt_buckets if row[key] in hot_keys else 0

    def bucket(k, salt):
        return zlib.crc32(f"{k}#{salt}".encode()) % n

    lparts = [[] for _ in range(n)]
    for l in left:
        s = row_salt(l)
        lparts[bucket(l[key], s)].append((s, l))
    rparts = [[] for _ in range(n)]
    for r in right:
        salts = range(salt_buckets) if r[key] in hot_keys else (0,)
        for s in salts:
            rparts[bucket(r[key], s)].append((s, r))

    outputs = []
    for lp, rp in zip(lparts, rparts):
        index = defaultdict(list)
        for s, r in rp:
            index[(r[key], s)].append(r)
        outputs.append([{**l, **r} for s, l in lp for r in index.get((l[key], s), [])])
    return [row for out in outputs for row in out], [len(o) for o in outputs]

**Explanation**

Salting splits one hot key into `salt_buckets` sub-keys spread over different partitions; the small side is replicated so every sub-key still finds its match. The cost is
`salt_buckets ×` more rows on the small side for hot keys only. Spark 3's **Adaptive Query Execution** (`spark.sql.adaptive.skewJoin.enabled`) can do a similar split automatically.

**Complexity:** O(n + m × salt_buckets for hot keys).

**Common mistakes**

- Salting both sides randomly → matches split up and rows are lost.
- Salting every key (replicating the whole small side ×8).